In [17]:
import multiprocessing
import mtkahypar
from pathlib import Path

# Initialize
mtk = mtkahypar.initialize(multiprocessing.cpu_count()) # use all available cores

# Setup partitioning context
context = mtk.context_from_preset(mtkahypar.PresetType.DEFAULT)
# In the following, we partition a hypergraph into two blocks
# with an allowed imbalance of 3% and optimize the connectivity metric
context.set_partitioning_parameters(
  2,                       # number of blocks
  0.03,                    # imbalance parameter
  mtkahypar.Objective.KM1) # objective function
mtkahypar.set_seed(42)     # seed
context.logging = True



 [WARNING]  Mt-KaHyPar is already initialized 


In [ ]:

# Load hypergraph from file (assumes hMetis file format per default)
hypergraph = mtk.hypergraph_from_file("path/to/hypergraph/file", context)

In [ ]:
# Partition hypergraph
partitioned_hg = hypergraph.partition(context)

# Output metrics
print("Partition Stats:")
print("Imbalance = " + str(partitioned_hg.imbalance(context)))
print("km1       = " + str(partitioned_hg.km1()))
print("Block Weights:")
for i in partitioned_hg.blocks():
  print(f"Weight of Block {i} = {partitioned_hg.block_weight(i)}")

In [1]:
import multiprocessing
import mtkahypar
import scipy.io as sio
import numpy as np
from pathlib import Path
import argparse
from sklearn.preprocessing import normalize
import tempfile


def mtx_to_hypergraph(mtx_file, context, mtk):
    """
    Convert a Matrix Market file to hypergraph representation for mtkahypar by
    creating a temporary hMetis format file.
    Row-wise partitioning: each row becomes a vertex, each column becomes a hyperedge.
    """
    # Load the MTX file using scipy
    mtx_path = Path(mtx_file)
    matrix = sio.mmread(mtx_path)
    
    # Convert to CSR format for efficient row access
    matrix_csr = matrix.tocsr()
    
    num_rows, num_cols = matrix.shape
    print(f"Matrix dimensions: {num_rows} rows x {num_cols} columns")
    
    # Create a temporary hMetis file
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.hgr') as temp_file:
        temp_path = temp_file.name
        
        # hMetis file format:
        # First line: <num_hyperedges> <num_vertices> <format> <weight_info>
        # Format: 0 = unweighted, 1 = weighted
        # Weight_info: 0 = no weights, 1 = edge weights, 2 = vertex weights, 3 = both
        
        # We'll use format=1 (weighted) and weight_info=3 (both edge and vertex weights)
        temp_file.write(f"{num_cols} {num_rows} 1 3\n")
        
        # For each hyperedge (column), write the vertices (rows) with non-zero entries
        for col in range(num_cols):
            # Get non-zero entries in this column
            col_data = matrix.getcol(col).nonzero()[0]
            
            # Write hyperedge weight (can be customized)
            edge_weight = 1  # Default weight
            
            # Write vertices (rows) in this hyperedge, adjusting for 1-based indexing in hMetis
            vertices = [str(row + 1) for row in col_data]
            
            # Write the hyperedge line: weight followed by vertices
            temp_file.write(f"{edge_weight} {' '.join(vertices)}\n")
        
        # Write vertex weights to the end of the file
        temp_file.write("\n")  # Separator line
        
        for row in range(num_rows):
            # Using nnz per row as weight - customize as needed
            vertex_weight = max(1, matrix_csr.getrow(row).nnz)  # Ensure minimum weight of 1
            temp_file.write(f"{vertex_weight}\n")
    
    print(f"Created temporary hMetis file: {temp_path}")
    
    # Load the hypergraph from the temporary file
    try:
        hypergraph = mtk.hypergraph_from_file(temp_path, context)
        print(f"Successfully loaded hypergraph: {hypergraph.num_hyperedges()} hyperedges, {hypergraph.num_vertices()} vertices")
        return hypergraph
    finally:
        # Clean up the temporary file
        import os
        try:
            os.remove(temp_path)
            print(f"Removed temporary file: {temp_path}")
        except Exception as e:
            print(f"Warning: Could not remove temporary file {temp_path}: {e}")


def create_parser():
    """Create a parser with fixed help strings that escape % characters properly."""
    parser = argparse.ArgumentParser(description='Partition a matrix in MTX format using mtkahypar')
    
    # Required arguments
    parser.add_argument('mtx_file', type=str, help='Path to the MTX file')
    
    # Optional arguments - carefully check each help string for % characters
    parser.add_argument('--blocks', type=int, default=2, 
                        help='Number of partitions')
    parser.add_argument('--imbalance', type=float, default=0.03, 
                        help='Allowed imbalance (e.g., 0.03 for 3%%)')  # Note the double %% to escape
    parser.add_argument('--preset', type=str, default='default', 
                        choices=['default', 'quality', 'highest_quality'], 
                        help='Preset type for partitioning')
    parser.add_argument('--seed', type=int, default=42, 
                        help='Random seed')
    parser.add_argument('--output', type=str, default=None, 
                        help='Output file for partition results')
    parser.add_argument('--block-weights', type=str, default=None, 
                        help='Comma-separated list of target block weights as fractions (must sum to 1)')
    parser.add_argument('--block-weights-file', type=str, default=None,
                        help='Path to a file containing block weights, one per line')
    
    return parser

In [2]:
path_mtx = Path("data/sample/in/A.mtx")
param_blocks = 16
param_imbal = 0.01
param_preset = mtkahypar.PresetType.HIGHEST_QUALITY
param_seed = 42
path_out = Path("data/sample/out/A_split.idk")
param_weights = np.array([
  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,
  2.352, 2.352, 2.352, 2.352, 2.352, 2.352, 2.352, 2.352
])
param_weights = normalize(param_weights.reshape((1,-1)))

In [3]:
mtk = mtkahypar.initialize(multiprocessing.cpu_count())
    
# Setup partitioning context
context = mtk.context_from_preset(param_preset)

In [4]:
context.set_partitioning_parameters(
    param_blocks,                  # number of blocks
    param_imbal,              # imbalance parameter
    mtkahypar.Objective.KM1)     # objective function

In [5]:
# Set seed and logging
mtkahypar.set_seed(param_seed)
context.logging = True

# Convert MTX to hypergraph
print(f"Loading and converting {path_mtx} to hypergraph...")
hypergraph = mtx_to_hypergraph(path_mtx, context, mtk)

: 

: 

: 

In [ ]:
def main():
    # Parse command line arguments
    parser = create_parser()
    args = parser.parse_args()
    # Map preset string to enum
    preset_map = {
        'default': mtkahypar.PresetType.DEFAULT,
        'quality': mtkahypar.PresetType.QUALITY,
        'highest_quality': mtkahypar.PresetType.HIGHEST_QUALITY
    }
    preset = preset_map[args.preset.lower()]
    
    # Initialize
    mtk = mtkahypar.initialize(multiprocessing.cpu_count())
    
    # Setup partitioning context
    context = mtk.context_from_preset(preset)
    
    # Get the number of blocks
    num_blocks = args.blocks
    
    # Set custom block weights if provided
    block_weights = None
    
    # Check if weights are provided via command line
    if args.block_weights:
        block_weights = [float(w) for w in args.block_weights.split(',')]
    
    # Check if weights are provided via file (file takes precedence)
    if args.block_weights_file:
        weights_path = Path(args.block_weights_file)
        if not weights_path.exists():
            raise FileNotFoundError(f"Block weights file not found: {args.block_weights_file}")
        
        # Read weights from file, one per line
        with open(weights_path, 'r') as f:
            lines = [line.strip() for line in f.readlines()]
            # Filter out empty lines and comments
            lines = [line for line in lines if line and not line.startswith('#')]
            block_weights = [float(line) for line in lines]
    
    # Apply block weights if provided
    if block_weights:
        if len(block_weights) != num_blocks:
            raise ValueError(f"Number of block weights ({len(block_weights)}) must match number of blocks ({num_blocks})")
        
        # Normalize weights to sum to 1.0
        weight_sum = sum(block_weights)
        if abs(weight_sum - 1.0) > 1e-6:
            print(f"Warning: Block weights sum to {weight_sum}, normalizing to 1.0")
            block_weights = [w / weight_sum for w in block_weights]
        
        # Apply custom block weights
        context.set_target_block_weights(block_weights)
        print(f"Using custom block weights: {block_weights}")
    
    # Set partitioning parameters
    context.set_partitioning_parameters(
        num_blocks,                  # number of blocks
        args.imbalance,              # imbalance parameter
        mtkahypar.Objective.KM1)     # objective function
    
    # Set seed and logging
    mtkahypar.set_seed(args.seed)
    context.logging = True
    
    # Convert MTX to hypergraph
    print(f"Loading and converting {args.mtx_file} to hypergraph...")
    hypergraph = mtx_to_hypergraph(args.mtx_file, context)
    
    # Partition hypergraph
    print(f"Partitioning into {num_blocks} blocks...")
    partitioned_hg = hypergraph.partition(context)
    
    # Output metrics
    print("\nPartition Stats:")
    print(f"Imbalance = {partitioned_hg.imbalance(context)}")
    print(f"km1       = {partitioned_hg.km1()}")
    print("\nBlock Weights:")
    for i in partitioned_hg.blocks():
        print(f"Weight of Block {i} = {partitioned_hg.block_weight(i)}")
    
    # Output partition assignment if requested
    if args.output:
        output_path = Path(args.output)
        with open(output_path, 'w') as f:
            for vertex in range(hypergraph.num_vertices()):
                f.write(f"{vertex} {partitioned_hg.block(vertex)}\n")
        print(f"\nPartition assignment written to {args.output}")
    
    # Print summary of partition assignment (how many vertices in each block)
    block_counts = {}
    for vertex in range(hypergraph.num_vertices()):
        block = partitioned_hg.block(vertex)
        block_counts[block] = block_counts.get(block, 0) + 1
    
    print("\nVertex Distribution:")
    for block, count in sorted(block_counts.items()):
        print(f"Block {block}: {count} vertices ({count/hypergraph.num_vertices()*100:.2f}%)")

if __name__ == "__main__":
    main()

In [ ]:
import multiprocessing
import mtkahypar

# Initialize
mtk = mtkahypar.initialize(multiprocessing.cpu_count()) # use all available cores

# Setup partitioning context
context = mtk.context_from_preset(mtkahypar.PresetType.DEFAULT)
# In the following, we partition a hypergraph into two blocks
# with an allowed imbalance of 3% and optimize the connectivity metric
context.set_partitioning_parameters(
  2,                       # number of blocks
  0.03,                    # imbalance parameter
  mtkahypar.Objective.KM1) # objective function
mtkahypar.set_seed(42)     # seed
context.logging = True

# Load hypergraph from file (assumes hMetis file format per default)
hypergraph = mtk.hypergraph_from_file("path/to/hypergraph/file", context)

# Partition hypergraph
partitioned_hg = hypergraph.partition(context)

# Output metrics
print("Partition Stats:")
print("Imbalance = " + str(partitioned_hg.imbalance(context)))
print("km1       = " + str(partitioned_hg.km1()))
print("Block Weights:")
for i in partitioned_hg.blocks():
  print(f"Weight of Block {i} = {partitioned_hg.block_weight(i)}")

# start here

In [1]:
import scipy.io
import scipy.sparse
import argparse
import sys
import os
from pathlib import Path
import mtkahypar
import numpy as np
from sklearn.preprocessing import normalize

def mtx_to_hmetis_rownet(mtx_filepath, hgr_filepath):
    """
    Converts a sparse matrix from Matrix Market (.mtx) format to
    hMetis (.hgr) format using the row-net hypergraph model.

    In the row-net model:
    - Each matrix ROW becomes a hyperedge.
    - Each matrix COLUMN becomes a vertex.
    - A vertex `j` is included in hyperedge `i` if matrix[i, j] is non-zero.

    Args:
        mtx_filepath (str): Path to the input Matrix Market file (.mtx).
        hgr_filepath (str): Path to the output hMetis file (.hgr).
    """
    print(f"Reading Matrix Market file: {mtx_filepath}")
    try:
        # Read the matrix market file
        # mmread automatically detects symmetry and field type (real, integer, pattern)
        # It returns a sparse matrix, typically in COO format if read from coordinate file
        matrix = scipy.io.mmread(mtx_filepath)

        # Ensure the matrix is in COO format for easy access to row/col indices
        if not isinstance(matrix, scipy.sparse.coo_matrix):
            matrix = matrix.tocoo()

        num_rows, num_cols = matrix.shape
        num_nonzeros = matrix.nnz # Number of explicitly stored non-zeros

        print(f"Matrix dimensions: {num_rows} rows (hyperedges), {num_cols} columns (vertices)")
        print(f"Number of non-zeros: {num_nonzeros}")

        # Check if the matrix is empty
        if num_rows == 0 or num_cols == 0:
             print("Warning: Matrix is empty. Generating an empty hMetis file.")
             with open(hgr_filepath, 'w') as f_out:
                  f_out.write("0 0\n") # Write header for empty hypergraph
             print(f"Empty hMetis file written to: {hgr_filepath}")
             return


        # --- Build the hypergraph structure (Row-Net Model) ---
        # hyperedges is a list where index `i` corresponds to matrix row `i` (0-based).
        # Each element `hyperedges[i]` will be a set of vertices (1-based column indices)
        # connected by hyperedge `i`. Using sets avoids duplicates automatically.
        hyperedges = [set() for _ in range(num_rows)]

        # Iterate through the non-zero entries (indices are 0-based in COO format)
        for r, c in zip(matrix.row, matrix.col):
            # Add the 1-based column index (vertex id) to the set for the corresponding row (hyperedge)
            hyperedges[r].add(c + 1)

        # --- Write the hMetis file ---
        print(f"Writing hMetis file: {hgr_filepath}")
        with open(hgr_filepath, 'w') as f_out:
            # Write the header: num_hyperedges (rows) num_vertices (cols)
            # Assuming unweighted format for now. Add 'fmt' code if weights are needed.
            f_out.write(f"{num_rows} {num_cols}\n")

            # Write each hyperedge (matrix row)
            for i in range(num_rows):
                # Get the set of vertices (1-based column indices) for this hyperedge
                vertex_set = hyperedges[i]

                if not vertex_set:
                    # Handle rows with no non-zeros (empty hyperedges)
                    # hMetis format typically expects vertices on the line.
                    # Writing a blank line might be acceptable, or skipping it.
                    # Check partitioner behavior if this occurs. For now, write blank line.
                    f_out.write("\n")
                    # Alternatively, you could skip writing the line if the partitioner handles it,
                    # but this would mismatch the hyperedge count in the header.
                    # print(f"Warning: Row {i+1} has no non-zeros (empty hyperedge). Writing blank line.")
                else:
                    # Convert set to sorted list for consistent output order
                    vertex_list = sorted(list(vertex_set))
                    # Write the space-separated list of 1-based vertex indices
                    f_out.write(" ".join(map(str, vertex_list)) + "\n")

        print("Conversion successful!")

    except FileNotFoundError:
        print(f"Error: Input file not found at {mtx_filepath}", file=sys.stderr)
        sys.exit(1)
    except Exception as e:
        print(f"An error occurred: {e}", file=sys.stderr)
        sys.exit(1)

# if __name__ == "__main__":
#     parser = argparse.ArgumentParser(
#         description="Convert a Matrix Market (.mtx) file to hMetis (.hgr) format using the row-net model.",
#         formatter_class=argparse.ArgumentDefaultsHelpFormatter
#     )
#     parser.add_argument("mtx_input", help="Path to the input Matrix Market file (.mtx)")
#     parser.add_argument("hgr_output", help="Path for the output hMetis file (.hgr)")

#     args = parser.parse_args()

#     # Basic input validation
#     if not os.path.exists(args.mtx_input):
#          print(f"Error: Input file '{args.mtx_input}' does not exist.", file=sys.stderr)
#          sys.exit(1)
#     if not args.mtx_input.lower().endswith(".mtx"):
#         print("Warning: Input file does not have a .mtx extension.", file=sys.stderr)
#     if not args.hgr_output.lower().endswith(".hgr"):
#         print("Warning: Output file does not have a .hgr extension.", file=sys.stderr)

#     mtx_to_hmetis_rownet(args.mtx_input, args.hgr_output)

In [2]:
numrows = 320

path_mtx = Path("data/sample320/in/A.mtx")
param_imbal = .1
param_preset = mtkahypar.PresetType.HIGHEST_QUALITY
param_seed = 42
path_hypergraph = Path("data/sample320/in/A.hMetis")
param_blocks = 16
path_out = Path(f"data/sample320/in/row_part_{param_blocks}")

target_weight = numrows + numrows*param_imbal
target_weight = int(target_weight)

# param_weights = np.array([1.0,1.0])
# param_weights = np.array([
#   1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,
#   2.352, 2.352, 2.352, 2.352, 2.352, 2.352, 2.352, 2.352
# ])
param_weights = np.array(
  [1.7]*8 +
  [1.0]*8
)
param_weights *=  target_weight/np.sum(param_weights)
param_weights = param_weights.astype(int)
for i in range(target_weight - np.sum(param_weights)):
  param_weights[-i-1] += 1
param_weights

array([27, 27, 27, 27, 27, 27, 27, 27, 17, 17, 17, 17, 17, 17, 17, 17])

In [3]:
mtx_to_hmetis_rownet(path_mtx, path_hypergraph)

Reading Matrix Market file: data/sample320/in/A.mtx
Matrix dimensions: 320 rows (hyperedges), 320 columns (vertices)
Number of non-zeros: 830
Writing hMetis file: data/sample320/in/A.hMetis
Conversion successful!


In [4]:
import multiprocessing
import mtkahypar

# Initialize
mtk = mtkahypar.initialize(multiprocessing.cpu_count()) # use all available cores

In [5]:
# Setup partitioning context
context = mtk.context_from_preset(mtkahypar.PresetType.HIGHEST_QUALITY)


# In the following, we partition a hypergraph into two blocks
# with an allowed imbalance of 3% and optimize the connectivity metric
context.set_partitioning_parameters(
  param_blocks,                       # number of blocks
  param_imbal,                    # imbalance parameter
  mtkahypar.Objective.KM1) # objective function
mtkahypar.set_seed(param_seed)     # seed
context.logging = True
context.set_individual_target_block_weights(param_weights.tolist())

# Load hypergraph from file (assumes hMetis file format per default)
hypergraph = mtk.hypergraph_from_file(str(path_hypergraph), context)

context.compute_max_block_weights(numrows)

[27, 27, 27, 27, 27, 27, 27, 27, 17, 17, 17, 17, 17, 17, 17, 17]

In [6]:
# Partition hypergraph
partitioned_hg = hypergraph.partition(context)

*******************************************************************************
*                            Partitioning Context                             *
*******************************************************************************
Partitioning Parameters:
  Hypergraph:                         
  Mode:                               direct_kway
  Objective:                          km1
  Gain Policy:                        km1
  Input File Format:                  hMetis
  Instance Type:                      hypergraph
  Preset Type:                        highest_quality
  Partition Type:                     n_level_hypergraph_partitioning
  k:                                  16
  epsilon:                            0.0731707
  seed:                               0
  Number of V-Cycles:                 0
  Ignore HE Size Threshold:           1000
  Large HE Size Threshold:            100
  Individual Part Weights:            27 27 27 27 27 27 27 27 17 17 17 17 17 17 17 17 
---

In [7]:
# Output metrics
print("Partition Stats:")
print("Imbalance = " + str(partitioned_hg.imbalance(context)))
print("km1       = " + str(partitioned_hg.km1()))
print("Block Weights:")
for i in partitioned_hg.blocks():
  print(f"Weight of Block {i} = {partitioned_hg.block_weight(i)}")

Partition Stats:
Imbalance = 0.0625
km1       = 155
Block Weights:
Weight of Block 0 = 25
Weight of Block 1 = 25
Weight of Block 2 = 26
Weight of Block 3 = 26
Weight of Block 4 = 26
Weight of Block 5 = 26
Weight of Block 6 = 26
Weight of Block 7 = 26
Weight of Block 8 = 16
Weight of Block 9 = 12
Weight of Block 10 = 17
Weight of Block 11 = 15
Weight of Block 12 = 15
Weight of Block 13 = 17
Weight of Block 14 = 12
Weight of Block 15 = 10


In [8]:
partitioned_hg.write_partition_to_file(str(path_out))

In [64]:
partitioned_hg.km1()

7

In [63]:
cut_value = partitioned_hg.cut()
print("Cut = " + str(cut_value))

Cut = 6


In [ ]:
help(context)

Help on Context in module mtkahypar object:

class Context(pybind11_builtins.pybind11_object)
 |  Method resolution order:
 |      Context
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  compute_max_block_weights(...)
 |      compute_max_block_weights(self: mtkahypar.Context, total_weight: int) -> list[int]
 |
 |
 |      Compute maximum allowed block weights for an input with the given total weight (depends on epsilon).
 |      If indiviual target weights are set, these are returned instead.
 |
 |      :param total_weight: Total weight of input hypergraph
 |
 |  compute_perfect_balance_block_weights(...)
 |      compute_perfect_balance_block_weights(self: mtkahypar.Context, total_weight: int) -> list[int]
 |
 |
 |      Compute block weights that represent perfect balance for an input with the given total weight.
 |      I

In [75]:
# context.epsilon = 0.9
# context.seed = 42
context.set_partitioning_parameters(
  param_blocks,                       # number of blocks
  param_imbal,                    # imbalance parameter
  mtkahypar.Objective.KM1) # objective function
context.print_configuration()

*******************************************************************************
*                            Partitioning Context                             *
*******************************************************************************
Partitioning Parameters:
  Hypergraph:                         
  Mode:                               direct_kway
  Objective:                          km1
  Gain Policy:                        none
  Input File Format:                  hMetis
  Preset Type:                        highest_quality
  Partition Type:                     UNDEFINED
  k:                                  16
  epsilon:                            0.1
  seed:                               0
  Number of V-Cycles:                 0
  Ignore HE Size Threshold:           1000
  Large HE Size Threshold:            4294967295
  Individual Part Weights:            138 138 138 138 138 138 138 138 325 325 325 325 325 325 325 325 
--------------------------------------------------------

In [56]:
np.sum(param_weights)/16

np.float64(231.5)

In [103]:
from os import system
from pathlib import Path

path_bin = Path("/software/kahypar/build/kahypar/application/KaHyPar")

system(f"{path_bin} --help")

+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ 
+                    _  __     _   _       ____                               + 
+                   | |/ /__ _| | | |_   _|  _ \ __ _ _ __                    + 
+                   | ' // _` | |_| | | | | |_) / _` | '__|                   + 
+                   | . \ (_| |  _  | |_| |  __/ (_| | |                      + 
+                   |_|\_\__,_|_| |_|\__, |_|   \__,_|_|                      + 
+                                    |___/                                    + 
+                 Karlsruhe Hypergraph Partitioning Framework                 + 
+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ 

Generic Options:
  --help                                show help message
  -v [ --verbose ] <bool>               Verbose main partitioning output
  --vip <bool>                          Verbose initial partitioning output
  -q [ --quiet ] <bool>                 Quiet 

0

In [121]:
path_base = Path("data/sample320/in/")
path_mtx = path_base / "A.mtx"
param_imbal = 0.03
param_preset = mtkahypar.PresetType.HIGHEST_QUALITY
param_seed = 42
path_hypergraph = path_base / "A.hMetis"
param_blocks = 16
path_out = path_base / "row_part_{param_blocks}"


param_weights = np.array(
  [1.0]*8 +
  [1.0]*8
)
param_weights = normalize(param_weights.reshape((1,-1)))
param_weights = (param_weights*1000).reshape((-1,)).astype(int)
param_weights

# cmd = f"""{path_bin} \
# -h {path_hypergraph} \
# -k {param_blocks} \
# -e {param_imbal} \
# -m recursive \
# -o km1 \
# --seed 42
# """
# print(cmd)

# system(cmd)



array([250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250, 250,
       250, 250, 250])

In [155]:
import kahypar
context = kahypar.Context()
context.loadINIconfiguration("km1_kKaHyPar_sea20.ini")

In [156]:
context.setEpsilon(param_imbal)
context.setK(param_blocks)
context.setSeed(param_seed)
# context.setCustomTargetBlockWeights(param_weights)

hypergraph = kahypar.createHypergraphFromFile(str(path_hypergraph),param_blocks)

In [ ]:
kahypar.partition(hypergraph, context)

: 

: 

: 

In [124]:
help(kahypar.createHypergraphFromFile)

Help on built-in function createHypergraphFromFile in module kahypar:

createHypergraphFromFile(...) method of builtins.PyCapsule instance
    createHypergraphFromFile(filename: str, k: int) -> kahypar.Hypergraph

    Construct a hypergraph from a file in hMETIS format

